In [0]:
# %pip install --upgrade typing_extensions transformers torch

In [0]:
from pyspark.sql import SparkSession
from transformers import pipeline
from pyspark.sql.functions import to_date, from_unixtime, col, first, last, max, min, avg, sum, count, when, udf, date_format, hour, expr, minute, floor, concat, lpad, lit
from pyspark.sql.functions import monotonically_increasing_id, coalesce, pandas_udf
from pyspark.sql.types import StructType, StructField, StringType, FloatType


In [0]:
%sql
SELECT * FROM `nvda_data` LIMIT 5

_c0,Date,Time,open,high,low,close,volume,RSI_rsi,SMA_sma,EMA_ema,MACD_macd,MACD_macd_signal,MACD_macd_hist,STOCH_slow_k,STOCH_slow_d,ADX_adx,CCI_cci,ATR_atr,BBANDS_upper_band,BBANDS_middle_band,BBANDS_lower_band
0,2023-05-22,09:30,30.901,31.52,30.68,31.39513,145468850.0,65.79526,31.31563,31.27555,0.47700234,0.54724325,-0.070240904,62.29355,64.44869,40.40849,26.16789,0.3571965,32.24575,31.029557,29.81337
1,2023-05-22,10:30,31.395,31.47,31.232,31.35269,48675280.0,64.5589,31.27948,31.29098,0.45342475,0.52847955,-0.0750548,58.59662,61.15217,37.53837,42.34475,0.3486825,32.18723,31.10803,30.02883
2,2023-05-22,11:30,31.354,31.3683,31.183,31.239,38064010.0,61.23924,31.28781,31.28058,0.42071598,0.50692683,-0.086210854,48.69357,56.52791,35.0044653,23.46293,0.33701252,32.12167,31.17018,30.21869
3,2023-05-22,12:30,31.236,31.304,31.15,31.2805,32525600.0,62.0071993,31.30287,31.28057,0.39360533,0.48426253,-0.0906572,52.30835,53.19951,32.7656,11.077173,0.32394026,32.031641,31.23193,30.43222
4,2023-05-22,13:30,31.285,31.399,31.284,31.367,24437720.0,63.62503,31.30076,31.29785,0.37477972,0.46236597,-0.087586252,59.84337,53.6151,30.52,30.64264,0.30926594,31.90152,31.29568,30.68985


In [0]:
%sql
SELECT COUNT(*) FROM `nvda_data`

count(1)
6920


In [0]:
%sql
SELECT * FROM `nvda_news` LIMIT 5

category,datetime,headline,id,image,related,source,summary,url
company,1717543740,Time to Buy the Post Earnings Dip in Dell Technologies (DELL) Stock,128084411,https://media.zenfs.com/en/zacks.com/c1cd358e97afad5e5afc22d86c6e3093,NVDA,Yahoo,"After reporting Q1 results after market hours last Thursday and falling -18% on Friday for what was its worst trading day since 2018, the post-earnings dip in Dell Technologies (DELL) stock is starting to look like a buying opportunity.",https://finnhub.io/api/news?id=ce26b8f466874da03aeccbe7a4051768463bbbacbefcbbb1dd860b45fe6b66ef
company,1717542183,"Exclusive-Chinese AI chip firms downgrading designs to secure TSMC production, sources say",128084412,https://media.zenfs.com/en/reuters-finance.com/2274046cf058aeb77efb8df5507b6d51,NVDA,Yahoo,"Some Chinese AI chip companies are now designing less powerful processors to retain access to Taiwan Semiconductor Manufacturing Co (TSMC) production in the face of U.S. sanctions, four people with knowledge of the matter said. Aiming to impede breakthroughs in artificial intelligence and supercomputing by China's military, Washington has imposed a series of export controls on highly sophisticated processors from companies such as Nvidia and on chip manufacturing equipment.",https://finnhub.io/api/news?id=009a34db2de8fdc87d6a1c5aa2830ce68889f07aeb378a43966e592373ac2e22
company,1717535944,S.Korean shares rise as Samsung Electronics jumps on Nvidia comments,128082498,null,NVDA,Finnhub,"Round-up of South Koreanfinancial markets: ** South Korean shares rose on Wednesday, led by heavyweightchipmaker Samsung Electronics after Nvidia's comments easedinvestor worries...",https://finnhub.io/api/news?id=2420f395c8aef93e0a849204448a8e234c74647fcbc0e2e334f47a4c862b1001
company,1717533111,Musk Explains Away Diverting Nvidia AI Chips From Tesla to X,128082888,https://s.yimg.com/ny/api/res/1.2/LVYnkobavzz_TJAQfr7OOQ--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD04MDA-/https://media.zenfs.com/en/bloomberg_markets_842/1d141355ec0e996e238d2ade7eac0005,NVDA,Yahoo,"(Bloomberg) -- Elon Musk confirmed he diverted artificial intelligence chips away from Tesla Inc. to his X Corp. and xAI Corp. ventures, offering explanations both for the redirected shipment and internal Nvidia Corp. emails casting doubt on the carmaker’s procurement plans.Most Read from BloombergModi Vows to Retain Power Even as Party Loses India MajorityShort Sellers in Danger of Extinction After Crushing Stock GainsBonds Rally as Traders Reload Fed Bets After Data: Markets WrapModi’s Magic I",https://finnhub.io/api/news?id=234f7a698b00a0c09bdb993d62ca71d4a6195af53572a1292ea716854b964dc1
company,1717532823,"Dow Jones, Other Indexes Close Near Session Highs; Cruise Lines Sail Higher While Nvidia Hits Record Highs",128077103,https://media.zenfs.com/en/ibd.com/088ed95138caca82aca3e2b370a9fda3,NVDA,Yahoo,The Dow Jones Industrial Average lost steam but led the major indexes on Tuesday. Carnival stock popped and lifted other cruise line stocks.,https://finnhub.io/api/news?id=65990fb9c1a0afcf3a3dad13dd203ec7f885f695961802b6e5efa7ce06963e99


In [0]:
%sql
SELECT COUNT(*) FROM `nvda_news`

count(1)
16670


In [0]:
df_prices = spark.sql("SELECT * FROM nvda_data")
display(df_prices)

In [0]:
df_prices = df_prices.drop('_c0', 'RSI_rsi', 'SMA_sma', 'EMA_ema', 'MACD_macd', 'MACD_macd_signal', 'MACD_macd_hist', 'STOCH_slow_k', 'STOCH_slow_d', 'ADX_adx', 'CCI_cci', 'ATR_atr', 'BBANDS_upper_band', 'BBANDS_middle_band', 'BBANDS_lower_band')
display(df_prices)

In [0]:
df_events = spark.sql("SELECT * FROM nvda_news")
display(df_events)

In [0]:

df = df_events.withColumn("DateTime", from_unixtime(col("datetime")))
df = df.drop('category', 'id', 'image', 'source', 'url', 'related') # remove related when multiple tickers being used
display(df)


In [0]:
df = df.withColumn("Date", to_date(col("DateTime")))
df = df.withColumn(
    "Time",
    concat(
        lpad(hour(col("DateTime")).cast("string"), 2, "0"),
        lit(":"),
        lpad((floor(minute(col("DateTime")) / 30) * 30).cast("int").cast("string"), 2, "0")
    )
)
df = df.drop('DateTime')
display(df)

In [0]:
from pyspark.sql.functions import abs, col, mean, stddev

# Add relative gap column
df_with_gap = df_prices.withColumn("daily_gap", abs(col("open") - col("close")) / col("open"))

# Calculate mean and stddev
gap_stats = df_with_gap.select(
    mean("daily_gap").alias("mean_gap"),
    stddev("daily_gap").alias("stddev_gap")
).collect()[0]

mean_gap = gap_stats["mean_gap"]
stddev_gap = gap_stats["stddev_gap"]
threshold = mean_gap + 2 * stddev_gap

# Filter abnormally large gaps
df_outliers = df_with_gap.filter(col("daily_gap") > threshold)


In [0]:
print(threshold)

0.014986004700278232


In [0]:
from pyspark.sql.functions import to_date, lit

# Ensure column is in date format
df_outliers = df_outliers.withColumn("Date", to_date("Date"))

# Filter rows
df_outliers = df_outliers.filter(df_outliers["Date"] >= lit("2024-06-01"))

In [0]:
display(df_outliers)

In [0]:
display(df)

In [0]:
df_tagged_events = df.join(
    df_outliers,
    on=["Date", "Time"],
    how="inner"
)

In [0]:
df_outliers.columns

Out[225]: ['Date', 'Time', 'open', 'high', 'low', 'close', 'volume', 'daily_gap']

In [0]:
df_tagged_events.createOrReplaceTempView("df_sentiment")

result = spark.sql("SELECT COUNT(*) FROM df_sentiment")
result.show()


+--------+
|count(1)|
+--------+
|     427|
+--------+



In [0]:
display(df_tagged_events)

In [0]:
def analyze_sentiment(text):
    if not text:
        return ("NEUTRAL", 0.0)
    
    from transformers import pipeline
    sentiment_pipeline = pipeline("sentiment-analysis", model="yiyanghkust/finbert-tone")
    
    # Call correctly
    result = sentiment_pipeline(text)[0]  # ✅ CORRECT CALL
    return (result["label"], float(result["score"]))

# Register UDF with schema
schema = StructType([
    StructField("label", StringType(), True),
    StructField("score", FloatType(), True)
])

sentiment_udf = udf(analyze_sentiment, schema)

In [0]:
df_with_sentiment = df_tagged_events.withColumn("sentiment_struct", sentiment_udf("summary"))
df_with_sentiment = df_with_sentiment.withColumn("sentiment", df_with_sentiment["sentiment_struct.label"])
df_with_sentiment = df_with_sentiment.withColumn("sentiment_score", df_with_sentiment["sentiment_struct.score"])
df_with_sentiment = df_with_sentiment.drop("sentiment_struct")


In [0]:
df_with_sentiment.createOrReplaceTempView("df_sentiment")

result = spark.sql("SELECT COUNT(*) FROM df_sentiment")
result.show()


+--------+
|count(1)|
+--------+
|     427|
+--------+



In [0]:
result = spark.sql("SELECT * FROM df_sentiment")
result.show()

In [0]:
display(result)

Date,Time,headline,summary,open,high,low,close,volume,daily_gap,sentiment,sentiment_score
2024-06-06,09:30,Chinese companies said to rent Nvidia GPUs from Oracle to bypass sanctions: report,Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,124.091003,125.587,118.4,121.089001,281920950.0,0.024191939201265094,Neutral,0.9999907
2024-06-06,09:30,Nvidia: Another Strong Quarter Reaffirms Investment Case,"Nvidia Corporation's strong growth continues, supported by supply constraints in the market, largely derisking the investment case. Read more on NVDA stock here.",124.091003,125.587,118.4,121.089001,281920950.0,0.024191939201265094,Positive,0.99999976
2024-06-06,09:30,"Taiwan Semi likely to be included in U.S. antitrust probe of Nvidia, Microsoft, OpenAi- CNBC",Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,124.091003,125.587,118.4,121.089001,281920950.0,0.024191939201265094,Neutral,0.9999907
2024-06-06,09:30,Hewlett Packard Enterprise Has A Long AI Runway (Rating Upgrade),"Hewlett Packard Enterprise benefits from expanding AI infrastructure, with improved revenue growth and margins expected. See why I'm upgrading HPE stock to Buy.",124.091003,125.587,118.4,121.089001,281920950.0,0.024191939201265094,Positive,1.0
2024-06-10,10:30,Nvidia growing to a 15% weight of the S&P 500 is ‘not out of the question’ - Evercore ISI,Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,119.92,122.47,119.56,122.3083,52422800.0,0.019915777184789867,Neutral,0.9999907
2024-06-10,10:30,"Commit To Buy NVIDIA At $6, Earn 79.2% Using Options",Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,119.92,122.47,119.56,122.3083,52422800.0,0.019915777184789867,Neutral,0.9999907
2024-06-10,10:30,"Nasdaq, S&P, Dow are mixed with Fed’s rate decision and CPI looming",Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,119.92,122.47,119.56,122.3083,52422800.0,0.019915777184789867,Neutral,0.9999907
2024-06-14,09:30,"As Apple, Nvidia Trade At All-Time High, Jim Cramer Tells Investors To Cash In On AI Stocks: 'Let's Not Be Too Greedy'",Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,129.94,132.75,128.32001,132.63969,96646990.0,0.020776435277820565,Neutral,0.9999907
2024-06-14,09:30,"Market Clubhouse Morning Memo - June 14th, 2024 (Trade Strategy For SPY, QQQ, AAPL, MSFT, NVDA, GOOGL, META And TSLA)",Looking for stock market analysis and research with proves results? Zacks.com offers in-depth financial research with over 30years of proven results.,129.94,132.75,128.32001,132.63969,96646990.0,0.020776435277820565,Neutral,0.9999907
2024-06-14,09:30,"Intel: Turnaround Is Working, I'm Bullish","Intel's stock may have dropped 40% YTD, but signs of a turnaround are evident with revenue growth and strategic investments.",129.94,132.75,128.32001,132.63969,96646990.0,0.020776435277820565,Positive,0.98975384


In [0]:
result.write.mode("overwrite").option("header", True).csv("dbfs:/FileStore/")